# Multi-Task Cardiologist Specialist GPU Training
### MIMIC-IV Clinical + Notes + ED Unified Dataset (101,008 Encounters)

This notebook fine-tunes **PubMedBERT** (`microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract`) for multi-task clinical cardiology inference:
1. **Disease Identification** (6 classes: `angina`, `arrhythmia`, `atrial fibrillation`, `coronary artery disease`, `heart failure`, `hypertension`)
2. **Chief Complaint Prediction** (11 classes: `cardiac assessment`, `chest pain`, `cough`, `dyspnea`, `edema`, `fatigue`, `headache`, `hypertensive symptoms`, `nausea`, `palpitations`, `syncope`)
3. **Clinical Urgency Triage** (3 classes: `Emergency`, `Urgent`, `Routine`)

**Scientific Guarantees**: 0 patient overlap across splits, 0 text overlap across splits, 0 target label leakage.

In [ ]:
# 1. Verify GPU Allocation & Install Dependencies
!nvidia-smi
!pip install -q transformers torch scikit-learn seaborn matplotlib onnx onnxruntime

In [ ]:
# 2. Unpack Dataset Splits
import os, zipfile

os.makedirs('datasets/processed/common', exist_ok=True)

# If you uploaded mimic_cardiologist_splits.zip directly to Colab root or Drive
if os.path.exists('mimic_cardiologist_splits.zip'):
    print("Unzipping uploaded mimic_cardiologist_splits.zip...")
    with zipfile.ZipFile('mimic_cardiologist_splits.zip', 'r') as z:
        z.extractall('datasets/processed/common')
elif os.path.exists('/content/drive/MyDrive/mimic_cardiologist_splits.zip'):
    print("Unzipping from Google Drive...")
    with zipfile.ZipFile('/content/drive/MyDrive/mimic_cardiologist_splits.zip', 'r') as z:
        z.extractall('datasets/processed/common')
elif not os.path.exists('datasets/processed/common/train_split.json'):
    print("Please upload mimic_cardiologist_splits.zip via the file upload prompt below:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as z:
                z.extractall('datasets/processed/common')

print("Datasets ready:")
for f in os.listdir('datasets/processed/common'):
    if f.endswith('.json'):
        print(f"  -> {f} ({os.path.getsize(os.path.join('datasets/processed/common', f))/(1024**2):.2f} MB)")

In [ ]:
# 3. Imports, Device & Seed Setup
import time, json, random, re
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt, seaborn as sns
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running training on device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# 4. Class Definitions & Multi-Task Dataset Class
DISEASE_CLASSES = ['angina', 'arrhythmia', 'atrial fibrillation', 'coronary artery disease', 'heart failure', 'hypertension']
COMPLAINT_CLASSES = ['cardiac assessment', 'chest pain', 'cough', 'dyspnea', 'edema', 'fatigue', 'headache', 'hypertensive symptoms', 'nausea', 'palpitations', 'syncope']
URGENCY_CLASSES = ['Emergency', 'Urgent', 'Routine']

DISEASE_MAP = {lbl: idx for idx, lbl in enumerate(DISEASE_CLASSES)}
COMPLAINT_MAP = {lbl: idx for idx, lbl in enumerate(COMPLAINT_CLASSES)}
URGENCY_MAP = {lbl: idx for idx, lbl in enumerate(URGENCY_CLASSES)}

class CardiologistMIMICDataset(Dataset):
    def __init__(self, records, tokenizer, max_len=256):
        self.records = records
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        self.texts = [r.get('text', 'No clinical narrative.') for r in records]
        self.diseases = [DISEASE_MAP.get(r.get('cardio_disease_labels', ['hypertension'])[0], 0) for r in records]
        self.complaints = [COMPLAINT_MAP.get(r.get('chief_complaint', 'cardiac assessment'), 0) for r in records]
        self.urgencies = [URGENCY_MAP.get(r.get('urgency', 'Emergency'), 0) for r in records]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'disease_label': torch.tensor(self.diseases[idx], dtype=torch.long),
            'complaint_label': torch.tensor(self.complaints[idx], dtype=torch.long),
            'urgency_label': torch.tensor(self.urgencies[idx], dtype=torch.long)
        }

In [ ]:
# 5. Multi-Task Model Architecture (PubMedBERT + 3 Heads)
class CardiologistMultiTaskClassifier(nn.Module):
    def __init__(self, model_name='microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract', num_diseases=6, num_complaints=11, num_urgencies=3):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_size = self.transformer.config.hidden_size
        
        self.fc_dis1 = nn.Linear(hidden_size, 256)
        self.fc_dis2 = nn.Linear(256, num_diseases)
        
        self.fc_comp1 = nn.Linear(hidden_size, 256)
        self.fc_comp2 = nn.Linear(256, num_complaints)
        
        self.fc_urg1 = nn.Linear(hidden_size, 256)
        self.fc_urg2 = nn.Linear(256, num_urgencies)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs[0][:, 0, :] # [CLS]
        
        d = self.dropout(self.relu(self.fc_dis1(cls_rep)))
        disease_logits = self.fc_dis2(d)
        
        c = self.dropout(self.relu(self.fc_comp1(cls_rep)))
        complaint_logits = self.fc_comp2(c)
        
        u = self.dropout(self.relu(self.fc_urg1(cls_rep)))
        urgency_logits = self.fc_urg2(u)
        
        return disease_logits, complaint_logits, urgency_logits

In [ ]:
# 6. Tokenize Splits & Configure Class Weights
data_dir = 'datasets/processed/common'
with open(f"{data_dir}/train_split.json", 'r') as f: train_records = json.load(f)
with open(f"{data_dir}/val_split.json", 'r') as f: val_records = json.load(f)
with open(f"{data_dir}/test_split.json", 'r') as f: test_records = json.load(f)

print(f"Loaded verified patient splits: Train={len(train_records)}, Val={len(val_records)}, Test={len(test_records)}")

MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = CardiologistMIMICDataset(train_records, tokenizer, max_len=256)
val_dataset = CardiologistMIMICDataset(val_records, tokenizer, max_len=256)
test_dataset = CardiologistMIMICDataset(test_records, tokenizer, max_len=256)

# Class weights for severe imbalance
dis_counts = np.bincount(train_dataset.diseases, minlength=6)
dis_weights = len(train_dataset.diseases) / (6.0 * np.maximum(dis_counts, 1).astype(float))
dis_weights_tensor = torch.tensor(dis_weights, dtype=torch.float).to(DEVICE)

urg_counts = np.bincount(train_dataset.urgencies, minlength=3)
urg_weights = len(train_dataset.urgencies) / (3.0 * np.maximum(urg_counts, 1).astype(float))
urg_weights_tensor = torch.tensor(urg_weights, dtype=torch.float).to(DEVICE)

criterion_dis = nn.CrossEntropyLoss(weight=dis_weights_tensor)
criterion_comp = nn.CrossEntropyLoss()
criterion_urg = nn.CrossEntropyLoss(weight=urg_weights_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, pin_memory=True)

In [ ]:
# 7. Execute Multi-Task Fine-Tuning with Mixed Precision
EPOCHS = 3
LR = 2e-5
OUTPUT_DIR = 'experiments/cardiologist_mimic'
os.makedirs(OUTPUT_DIR, exist_ok=True)

model = CardiologistMultiTaskClassifier(MODEL_NAME).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

best_val_f1 = -1.0
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        d_t = batch['disease_label'].to(DEVICE)
        c_t = batch['complaint_label'].to(DEVICE)
        u_t = batch['urgency_label'].to(DEVICE)
        
        optimizer.zero_grad()
        
        if torch.cuda.is_available():
            with torch.amp.autocast('cuda'):
                d_l, c_l, u_l = model(ids, mask)
                loss = criterion_dis(d_l, d_t) + 0.3 * criterion_comp(c_l, c_t) + 0.2 * criterion_urg(u_l, u_t)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            d_l, c_l, u_l = model(ids, mask)
            loss = criterion_dis(d_l, d_t) + 0.3 * criterion_comp(c_l, c_t) + 0.2 * criterion_urg(u_l, u_t)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
        scheduler.step()
        total_loss += loss.item()
        
        if (step + 1) % 500 == 0 or step == len(train_loader) - 1:
            print(f"Epoch {epoch+1}/{EPOCHS} | Step {step+1}/{len(train_loader)} | Batch Loss: {loss.item():.4f}")
            
    # Validation Loop
    model.eval()
    d_preds, d_trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            d_l, _, _ = model(ids, mask)
            d_preds.extend(torch.argmax(d_l, dim=1).cpu().numpy().tolist())
            d_trues.extend(batch['disease_label'].numpy().tolist())
            
    _, _, val_f1, _ = precision_recall_fscore_support(d_trues, d_preds, average='macro', zero_division=0)
    print(f"=== Epoch {epoch+1} Completed | Val Disease Macro-F1: {val_f1:.4f} ===")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_cardiologist_model.pt")
        print(f"--> Saved new best checkpoint to {OUTPUT_DIR}/best_cardiologist_model.pt")

print(f"\nTraining finished in {(time.time() - start_time)/60:.2f} minutes.")

In [ ]:
# 8. Comprehensive Test Split Evaluation
ckpt_path = f"{OUTPUT_DIR}/best_cardiologist_model.pt"
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()

d_preds, d_trues, d_probs = [], [], []
c_preds, c_trues, c_probs = [], [], []
u_preds, u_trues, u_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        d_l, c_l, u_l = model(ids, mask)
        
        d_preds.extend(torch.argmax(d_l, dim=1).cpu().numpy().tolist())
        d_trues.extend(batch['disease_label'].numpy().tolist())
        d_probs.extend(torch.softmax(d_l, dim=1).cpu().numpy().tolist())
        
        c_preds.extend(torch.argmax(c_l, dim=1).cpu().numpy().tolist())
        c_trues.extend(batch['complaint_label'].numpy().tolist())
        c_probs.extend(torch.softmax(c_l, dim=1).cpu().numpy().tolist())
        
        u_preds.extend(torch.argmax(u_l, dim=1).cpu().numpy().tolist())
        u_trues.extend(batch['urgency_label'].numpy().tolist())
        u_probs.extend(torch.softmax(u_l, dim=1).cpu().numpy().tolist())

print("\n=======================================================")
print("HEAD 1: DISEASE CLASSIFICATION REPORT (6 CLASSES)")
print("=======================================================")
print(classification_report(d_trues, d_preds, labels=range(len(DISEASE_CLASSES)), target_names=DISEASE_CLASSES, zero_division=0))

print("\n=======================================================")
print("HEAD 2: CHIEF COMPLAINT CLASSIFICATION REPORT (11 CLASSES)")
print("=======================================================")
print(classification_report(c_trues, c_preds, labels=range(len(COMPLAINT_CLASSES)), target_names=COMPLAINT_CLASSES, zero_division=0))

print("\n=======================================================")
print("HEAD 3: CLINICAL URGENCY CLASSIFICATION REPORT (3 CLASSES)")
print("=======================================================")
print(classification_report(u_trues, u_preds, labels=range(len(URGENCY_CLASSES)), target_names=URGENCY_CLASSES, zero_division=0))

In [ ]:
# 9. Calibration (ECE) & Confusion Matrices Plotting
def compute_ece(probs, labels, n_bins=10):
    probs, labels = np.array(probs), np.array(labels)
    preds, confs = np.argmax(probs, axis=1), np.max(probs, axis=1)
    ece, bin_boundaries = 0.0, np.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        in_bin = (confs > bin_boundaries[i]) & (confs <= bin_boundaries[i+1])
        if np.mean(in_bin) > 0:
            ece += np.mean(in_bin) * np.abs(np.mean(preds[in_bin] == labels[in_bin]) - np.mean(confs[in_bin]))
    return float(ece)

d_ece = compute_ece(d_probs, d_trues)
c_ece = compute_ece(c_probs, c_trues)
u_ece = compute_ece(u_probs, u_trues)

print(f"Expected Calibration Error (ECE):")
print(f"  Disease Head ECE: {d_ece:.4f}")
print(f"  Complaint Head ECE: {c_ece:.4f}")
print(f"  Urgency Head ECE: {u_ece:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(confusion_matrix(d_trues, d_preds, labels=range(6)), annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=DISEASE_CLASSES, yticklabels=DISEASE_CLASSES)
axes[0].set_title(f'Disease Head (ECE: {d_ece:.3f})')
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(confusion_matrix(c_trues, c_preds, labels=range(11)), annot=True, fmt='d', cmap='Oranges', ax=axes[1], xticklabels=[c[:6] for c in COMPLAINT_CLASSES], yticklabels=[c[:6] for c in COMPLAINT_CLASSES])
axes[1].set_title(f'Chief Complaint (ECE: {c_ece:.3f})')

sns.heatmap(confusion_matrix(u_trues, u_preds, labels=range(3)), annot=True, fmt='d', cmap='Greens', ax=axes[2], xticklabels=URGENCY_CLASSES, yticklabels=URGENCY_CLASSES)
axes[2].set_title(f'Clinical Urgency (ECE: {u_ece:.3f})')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
# 10. ONNX Export & INT8 Quantization
class CardiologistMultiTaskONNX(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.transformer = trained_model.transformer
        self.w_dis1 = nn.Parameter(trained_model.fc_dis1.weight.t().clone())
        self.b_dis1 = nn.Parameter(trained_model.fc_dis1.bias.clone())
        self.w_dis2 = nn.Parameter(trained_model.fc_dis2.weight.t().clone())
        self.b_dis2 = nn.Parameter(trained_model.fc_dis2.bias.clone())
        
        self.w_comp1 = nn.Parameter(trained_model.fc_comp1.weight.t().clone())
        self.b_comp1 = nn.Parameter(trained_model.fc_comp1.bias.clone())
        self.w_comp2 = nn.Parameter(trained_model.fc_comp2.weight.t().clone())
        self.b_comp2 = nn.Parameter(trained_model.fc_comp2.bias.clone())
        
        self.w_urg1 = nn.Parameter(trained_model.fc_urg1.weight.t().clone())
        self.b_urg1 = nn.Parameter(trained_model.fc_urg1.bias.clone())
        self.w_urg2 = nn.Parameter(trained_model.fc_urg2.weight.t().clone())
        self.b_urg2 = nn.Parameter(trained_model.fc_urg2.bias.clone())

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs[0][:, 0, :].reshape(-1, self.transformer.config.hidden_size)
        
        d = torch.matmul(cls_rep, self.w_dis1) + self.b_dis1
        logits_disease = torch.matmul(torch.relu(d), self.w_dis2) + self.b_dis2
        
        c = torch.matmul(cls_rep, self.w_comp1) + self.b_comp1
        logits_complaint = torch.matmul(torch.relu(c), self.w_comp2) + self.b_comp2
        
        u = torch.matmul(cls_rep, self.w_urg1) + self.b_urg1
        logits_urgency = torch.matmul(torch.relu(u), self.w_urg2) + self.b_urg2
        
        return logits_disease, logits_complaint, logits_urgency

onnx_model = CardiologistMultiTaskONNX(model).to(DEVICE)
onnx_model.eval()
dummy_ids = torch.ones(1, 256, dtype=torch.long).to(DEVICE)
dummy_mask = torch.ones(1, 256, dtype=torch.long).to(DEVICE)

onnx_path = f"{OUTPUT_DIR}/cardiologist_mimic_multitask.onnx"
torch.onnx.export(
    onnx_model,
    (dummy_ids, dummy_mask),
    onnx_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['disease_logits', 'complaint_logits', 'urgency_logits'],
    dynamic_axes={'input_ids': {0: 'batch_size', 1: 'sequence_length'}, 'attention_mask': {0: 'batch_size', 1: 'sequence_length'}},
    opset_version=18
)
print(f"Saved ONNX model to {onnx_path}")

quant_path = f"{OUTPUT_DIR}/cardiologist_mimic_multitask_quant.onnx"
quantize_dynamic(model_input=onnx_path, model_output=quant_path, weight_type=QuantType.QUInt8)
print(f"Saved INT8 Quantized ONNX to {quant_path}")

# Benchmark CPU Latency
sess = ort.InferenceSession(quant_path, providers=['CPUExecutionProvider'])
times = []
for batch in test_dataset.encodings[:20]:
    inp = batch['input_ids'].numpy().reshape(1, -1)
    msk = batch['attention_mask'].numpy().reshape(1, -1)
    t0 = time.time()
    sess.run(None, {'input_ids': inp, 'attention_mask': msk})
    times.append((time.time() - t0) * 1000)
print(f"Average ONNX CPU Inference Latency: {np.mean(times):.2f} ms/sample")

# Save full metrics JSON manifest
dp_m, dr_m, df_m, _ = precision_recall_fscore_support(d_trues, d_preds, average='macro', zero_division=0)
dp_w, dr_w, df_w, _ = precision_recall_fscore_support(d_trues, d_preds, average='weighted', zero_division=0)
cp_m, cr_m, cf_m, _ = precision_recall_fscore_support(c_trues, c_preds, average='macro', zero_division=0)
up_m, ur_m, uf_m, _ = precision_recall_fscore_support(u_trues, u_preds, average='macro', zero_division=0)

metrics_manifest = {
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'model': MODEL_NAME,
    'epochs': EPOCHS,
    'test_samples': len(d_trues),
    'disease': {'accuracy': float(np.mean(np.array(d_trues)==np.array(d_preds))), 'macro_f1': float(df_m), 'weighted_f1': float(df_w), 'ece': float(d_ece)},
    'complaint': {'accuracy': float(np.mean(np.array(c_trues)==np.array(c_preds))), 'macro_f1': float(cf_m), 'ece': float(c_ece)},
    'urgency': {'accuracy': float(np.mean(np.array(u_trues)==np.array(u_preds))), 'macro_f1': float(uf_m), 'ece': float(u_ece)},
    'onnx_cpu_latency_ms': float(np.mean(times))
}

with open(f"{OUTPUT_DIR}/cardiologist_mimic_metrics.json", 'w') as f:
    json.dump(metrics_manifest, f, indent=2)
print(f"Saved metrics JSON manifest to {OUTPUT_DIR}/cardiologist_mimic_metrics.json")

In [ ]:
# 11. Package Artifacts for 1-Click Download
!zip -r /content/cardiologist_mimic_experiment_results.zip experiments/cardiologist_mimic/
print("\nCreated /content/cardiologist_mimic_experiment_results.zip")
from google.colab import files
files.download('/content/cardiologist_mimic_experiment_results.zip')